# TD5 - File Handling in Python

### Resources

- news.csv: https://github.com/ApnaClassroom/Dataset/blob/main/News.csv
- recipes.json: https://www.kaggle.com/datasets/karthikrathod/json-files-of-datasets
- ner.txt: https://github.com/BIDS-Xu-Lab/section_specific_annotation_of_PICO/blob/main/data/EBM-NLPmod/fold1/train.txt
- expressions.cupt: https://gitlab.com/parseme/sharedtask-data/-/blob/master/2.0/subtask1/FR/train.cupt?ref_type=heads

## Exercise 1

- Take a look at the structure of the file "news_sample.csv"
- Without using any dedicated library, open and parse this file
- Print every fake news in this file

In [9]:
# Opening the file

with open("TD5-files/news_sample.csv","r",encoding="utf-8") as f:
    content = f.read()

# Step 1: extract the basic structure (line separation)
    
content = content.split("\n")

# Step 2: split the information in each line
# There are commas in the news text, we have to find a way to go around this issue
# Here, I replace ",TRUE" and ",FALSE" in order to replace commas with a easier to handle symbol

structured = [content[0].split(",")] # Adding the header

for line in content[1:]: # Everything but the header
    
    line = line.replace(",FALSE","||FALSE").replace(",TRUE","||TRUE")
    line_split = line.split("||")
    
    structured.append(line_split)
    
# Step 3: print only fake news

for line in structured[1:]:
    if line[1] == "FALSE":
        print(line)

['Says the Annies List political group supports third-trimester abortions on demand.', 'FALSE']
['Health care reform legislation is likely to mandate free sex change surgeries.', 'FALSE']
['Jim Dunnam has not lived in the district he represents for years now.', 'FALSE']


## Exercise 2

- The previous file was only a small sample of the larger file "news.csv", take a look at it
- Without using any dedicated library, open and parse this file
- We want to add an "id" field to each row: 
    - The first row should have an id of 0, the second row an id of 1, etc
    - The fields must be in this order: id,news,fake
- Delete every row not corresponding to a fake news
- Save the result in "fake_news.csv"

In [23]:
# Opening the file

with open("TD5-files/news.csv","r",encoding="utf-8") as f:
    content = f.read()

# Step 1: extract the basic structure (line separation)
    
content = content.split("\n")
structured = [content[0].split(",")] # Adding the header

for line in content[1:]: # Everything but the header
    
    line = line.replace(",FALSE","||FALSE").replace(",TRUE","||TRUE")
    line_split = line.split("||")
    
    structured.append(line_split)
    
# Step 2: add the "id" field

for i,line in enumerate(structured[1:]):
    
    structured[i+1] = [str(i)] + line
    
structured[0] = ["id"] + structured[0] # Do not forget the header

# Step 3: remove news which is true

fake_news = [structured[0]]

for line in structured[1:]:

    if len(line) == 3:
    
        if line[2] == "FALSE":
            fake_news.append(line)
        
# Step 4: save the result in a file

fake_news_save = [",".join(i) for i in fake_news]
fake_news_save = "\n".join(fake_news_save)

with open("TD5-files/fake_news.csv","w",encoding="utf-8") as f:
    f.write(fake_news_save)

## Exercise 3

- Take a look at the structure of the file "recipes.json" - hard to understand, right ?
- Take a look at the json library in Python (https://docs.python.org/3/library/json.html)
- Create a function to load a json file
- Load the content of "recipes.json" in a "recipes_data" variable
- Create another function to save some data in a json file - use the "indent" parameter and set it to "4"
- Save the content of "recipe_data" in "recipes_indented.json"
- Take a look at this new json file - better, right ?

In [25]:
import json


def open_json(path):
    "open a json file"
    with open(path,'r',encoding='utf-8') as f:
        data = json.load(f)
    return data

  
def write_json(path,data):
    "create a json file"
    with open(path,"w",encoding='utf-8') as f:
        json.dump(data,f,indent=4,ensure_ascii=False)


recipes_data = open_json("TD5-files/recipes.json")
write_json("TD5-files/recipes_indented.json",recipes_data)

## Exercise 4

- Load the content of "recipes_indented.json"
- Create a list containing every recipe's name - let's call it "names"
- Use the "random" library to generate a random number between 1 and the length of "names"
- Use this number to fetch an element from "names" (using the indexes)
- Print useful information to prepare this randomly chosen recipe !

In [30]:
import random

recipes_data = open_json("TD5-files/recipes.json")

names = []

for key,values in recipes_data.items():
    names.append(values["title"])
    
random_number = random.randrange(1,len(names))
random_name = names[random_number]

for key,values in recipes_data.items():
    
    if values["title"] == random_name:
        print("INGREDIENTS:")
        print("\n".join(values["ingredients"]))
        print()
        print("INSTRUCTIONS:")
        print("\n".join(values["instructions"]))
        break # If there are duplicates

INGREDIENTS:
2 teaspoons Italian seasoning
0.5 teaspoon seasoning salt
0.5 teaspoon mustard powder
1 teaspoon garlic powder
0.5 teaspoon ground black pepper
1 (3 pound) whole chicken
2 lemons
2 tablespoons olive oil

INSTRUCTIONS:
Preheat oven to 350 degrees F (175 degrees C).
Combine the seasoning, salt, mustard powder, garlic powder and black pepper; set aside. Rinse the chicken thoroughly, and remove the giblets. Place chicken in a 9x13 inch baking dish. Sprinkle 1 1/2 teaspoons of the spice mixture inside the chicken. Rub the remaining mixture on the outside of the chicken.
Squeeze the juice of the 2 lemons into a small bowl or cup, and mix with the olive oil. Drizzle this oil/juice mixture over the chicken.
Bake in the preheated oven for 1 1/2 hours, or until juices run clear, basting several times with the remaining oil mixture.


## Exercise 5

- Load "recipes_indented.json"
- Fetch every possible ingredient from all recipes and save them to a list - you will have a lot of noise, it does not need to be perfect !
- Try to take care of duplicates in this list
    - If you know about regex, do this step - If you don't, ignore it
- Print at least 50 of these ingredients
- Ask the user for one ingredient using the "input" call
- Fetch every recipe with this ingredient
- Create a new file with these recipes and save it as "recipes_ingredient.json"

In [45]:
import re

recipes_data = open_json("TD5-files/recipes.json")

all_ingredients = set()

for key,values in recipes_data.items():
    
    for ingredient in values["ingredients"]:
        
        ingredient = re.sub(r"\d|\.","",ingredient)
        ingredient = re.sub(r"\(.*\)","",ingredient)
        all_ingredients.add(ingredient)

all_ingredients = list(all_ingredients)

ingr_1 = input("First Ingredient: ")
recipes = []

for key,values in recipes_data.items():
    
    for ingredient in values["ingredients"]:
        
        if ingr_1 in ingredient:
            recipes.append(values)
            break
            
write_json(f"TD5-files/recipes_{ingr_1}.json",recipes)

First Ingredient: 


## Exercise 6

- Take a look at the structure of the file "ner.txt"
- Try to find a way to parse this data structure in Python (hint: "DOCTSTART" and empty lines are important)
- Save the result as "ner.json"
- Open "pico_ner.json" and extract every entity in it:
    - The first word of an entity has the label "B-" + the entity category (P,I or O here)
    - The following words belonging to this entity have the label "I-" + the entity category
- Save theses entities in "entities.txt", where each line correspond to an entity

In [55]:
with open("TD5-files/ner.txt","r",encoding="utf-8") as f:
    content = f.read()

content = content.split("\n\n")
content = [i.split("\n") for i in content]

content_structured = []

for line in content:
    line_structured = []
    
    for l in line:  
        l_split = l.split("\t")
        line_structured.append(l_split)
        
    content_structured.append(line_structured)

write_json("TD5-files/ner.json",content_structured)

## Bonus Challenge

- Take a look at the structure of the file expressions.cupt
- Try to find a way to parse this data structure in Python
- Save the result as "expressions.json"
- Bonus: extract every expression (MWE) from this corpus and save them in "expressions.txt"